<a href="https://colab.research.google.com/github/smriithhii/nlpr/blob/main/note_smr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


cleaning of data

kaavya update code

In [ ]:
import pandas as pd
import re

# Load datasets
file_path1 = "/content/drive/MyDrive/nlpr/3_Student depression and anxiety dataset.xlsx"
file_path2 = "/content/drive/MyDrive/nlpr/1_depression_dataset_reddit_cleaned.csv"
file_path3 = "/content/drive/MyDrive/nlpr/2_Stress.csv"
file_path4 = "/content/drive/MyDrive/nlpr/4_Suicide_Ideation_Dataset.csv"

df1 = pd.read_excel(file_path1)
df2 = pd.read_csv(file_path2)
df3 = pd.read_csv(file_path3)
df4 = pd.read_csv(file_path4, encoding='latin1')  # Added encoding='latin1'

# ---------------- CLEANING FUNCTIONS ---------------- #

def clean_text(text):
    """Clean text: lowercase, remove URLs, punctuation, numbers, and extra spaces."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)  # keep only alphabets and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

def clean_dataset(df, text_col='text', label_col='label'):
    """Clean dataset safely and drop rows with missing/empty text or labels."""
    df_temp = df.copy()

    # Identify actual columns
    actual_text_col = text_col if text_col in df_temp.columns else df_temp.columns[0]
    actual_label_col = label_col if label_col in df_temp.columns else None

    if not actual_label_col:
        print(f"⚠️ Warning: '{label_col}' column not found, skipping label processing.")

    # Clean text
    df_temp['cleaned_text'] = df_temp[actual_text_col].apply(clean_text)

    # Drop rows where either text or label is empty/NaN
    if actual_label_col:
        df_temp = df_temp.dropna(subset=[actual_text_col, actual_label_col])
        df_temp = df_temp[df_temp[actual_text_col].astype(str).str.strip() != '']
        df_temp = df_temp[df_temp[actual_label_col].astype(str).str.strip() != '']
        cleaned_df = df_temp[[actual_label_col, 'cleaned_text']]
    else:
        df_temp = df_temp.dropna(subset=[actual_text_col])
        df_temp = df_temp[df_temp[actual_text_col].astype(str).str.strip() != '']
        cleaned_df = df_temp[['cleaned_text']]

    print(f"✅ Cleaned {len(cleaned_df)} rows in {label_col if label_col else 'dataset without label'}")
    return cleaned_df

# ---------------- APPLY CLEANING ---------------- #

print("Cleaning Dataset 1:")
df1_cleaned = clean_dataset(df1)

print("\nCleaning Dataset 2:")
df2_cleaned = clean_dataset(df2, text_col='clean_text', label_col='is_depression')

print("\nCleaning Dataset 3:")
df3_cleaned = clean_dataset(df3, text_col='text', label_col='label')

print("\nCleaning Dataset 4:")
df4_cleaned = clean_dataset(df4, text_col='Tweet', label_col='Suicide')

# Convert Suicide labels: 'Not Suicide post' = 0, others = 1
df4_cleaned['Suicide'] = df4_cleaned['Suicide'].apply(lambda x: 0 if str(x).strip().lower() == 'not suicide post' else 1)

# ---------------- SAVE CLEANED VERSIONS ---------------- #

df1_cleaned.to_csv("/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset1.csv", index=False)
df2_cleaned.to_csv("/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset2.csv", index=False)
df3_cleaned.to_csv("/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset3.csv", index=False)
df4_cleaned.to_csv("/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset4.csv", index=False)

print("\n🎯 All cleaned datasets saved successfully!")

Cleaning Dataset 1:
✅ Cleaned 6970 rows in label

Cleaning Dataset 2:
✅ Cleaned 7731 rows in is_depression

Cleaning Dataset 3:
✅ Cleaned 2838 rows in label

Cleaning Dataset 4:
✅ Cleaned 1785 rows in Suicide

🎯 All cleaned datasets saved successfully!


Downsampling

In [ ]:
import pandas as pd

# File paths of cleaned datasets
paths = {
    "dataset1": "/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset1.csv",
    "dataset2": "/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset2.csv",
    "dataset3": "/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset3.csv",
    "dataset4": "/content/drive/MyDrive/nlpr/smr_nlp_testing/cleaned/dataset4.csv"
}

# Function to downsample dataset
def downsample_dataset(df, n=1300, label_col=None):
    if label_col and label_col in df.columns:
        # Stratified downsampling — keeps label ratio balanced
        df_down = (
            df.groupby(label_col, group_keys=False)
              .apply(lambda x: x.sample(
                  min(len(x), n // df[label_col].nunique()),
                  random_state=42))
              .reset_index(drop=True)
        )
    else:
        # Random downsampling if no label column
        df_down = df.sample(n=min(len(df), n), random_state=42).reset_index(drop=True)
    return df_down

# Loop through datasets and downsample
for name, path in paths.items():
    df = pd.read_csv(path)

    # Automatically detect label column
    possible_labels = ['label', 'is_depression', 'Suicide']
    label_col = next((col for col in possible_labels if col in df.columns), None)

    df_down = downsample_dataset(df, n=1300, label_col=label_col)

    # Save downsampled dataset
    out_path = f"/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/{name}_downsampled.csv"
    df_down.to_csv(out_path, index=False)

    print(f"✅ {name} downsampled to {len(df_down)} rows → saved at: {out_path}")


/tmp/ipython-input-202433319.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


✅ dataset1 downsampled to 1300 rows → saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset1_downsampled.csv


/tmp/ipython-input-202433319.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
/tmp/ipython-input-202433319.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


✅ dataset2 downsampled to 1300 rows → saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset2_downsampled.csv
✅ dataset3 downsampled to 1300 rows → saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset3_downsampled.csv
✅ dataset4 downsampled to 1300 rows → saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset4_downsampled.csv


/tmp/ipython-input-202433319.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


data aug - bert

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import random

device = 0 if torch.cuda.is_available() else -1

unmasker = pipeline('fill-mask', model='bert-base-uncased', device=device)

def augment_text_bert(text, num_augments=1):
    words = text.split()
    if len(words) < 3:
        return [text] * num_augments

    augmented_texts = []

    for _ in range(num_augments):
        num_masks = max(1, int(len(words) * 0.15))
        mask_positions = random.sample(range(len(words)), min(num_masks, len(words)))

        masked_text = words.copy()
        for pos in mask_positions:
            masked_text[pos] = '[MASK]'

        masked_sentence = ' '.join(masked_text)

        try:
            predictions = unmasker(masked_sentence)

            if isinstance(predictions[0], list):
                predictions = predictions[0]

            new_words = words.copy()
            for pos in mask_positions:
                new_words[pos] = predictions[0]['token_str'].strip()

            augmented_texts.append(' '.join(new_words))
        except:
            augmented_texts.append(text)

    return augmented_texts

def augment_dataset_bert(df, text_col='cleaned_text', label_col=None, aug_ratio=0.25):
    num_to_augment = int(len(df) * aug_ratio)

    sample_indices = random.sample(range(len(df)), num_to_augment)

    augmented_rows = []

    for idx in sample_indices:
        original_text = df.iloc[idx][text_col]
        augmented_text = augment_text_bert(original_text, num_augments=1)[0]

        new_row = df.iloc[idx].copy()
        new_row[text_col] = augmented_text
        augmented_rows.append(new_row)

    augmented_df = pd.DataFrame(augmented_rows)
    result_df = pd.concat([df, augmented_df], ignore_index=True)

    return result_df

paths = {
    "dataset1": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset1_downsampled.csv",
    "dataset2": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset2_downsampled.csv",
    "dataset3": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset3_downsampled.csv",
    "dataset4": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset4_downsampled.csv"
}

for name, path in paths.items():
    df = pd.read_csv(path)

    possible_labels = ['label', 'is_depression', 'Suicide']
    label_col = next((col for col in possible_labels if col in df.columns), None)

    print(f"Processing {name} for 25% BERT augmentation...")
    df_aug_25 = augment_dataset_bert(df, text_col='cleaned_text', label_col=label_col, aug_ratio=0.25)
    out_path_25 = f"/content/drive/MyDrive/nlpr/Augmented/BERT/{name}_bert_25.csv"
    df_aug_25.to_csv(out_path_25, index=False)
    print(f"✅ {name} augmented to {len(df_aug_25)} rows (25%) → saved at: {out_path_25}")

    print(f"Processing {name} for 50% BERT augmentation...")
    df_aug_50 = augment_dataset_bert(df, text_col='cleaned_text', label_col=label_col, aug_ratio=0.50)
    out_path_50 = f"/content/drive/MyDrive/nlpr/Augmented/BERT/{name}_bert_50.csv"
    df_aug_50.to_csv(out_path_50, index=False)
    print(f"✅ {name} augmented to {len(df_aug_50)} rows (50%) → saved at: {out_path_50}")

print("\n🎉 All datasets augmented with BERT (25% and 50%)!")

In [ ]:
dataset_mapping = {
    "dataset1": "d1",
    "dataset2": "d2",
    "dataset3": "d3",
    "dataset4": "d4"
}

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import random
import os

os.makedirs("/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25", exist_ok=True)
os.makedirs("/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/50", exist_ok=True)

device = 0 if torch.cuda.is_available() else -1

unmasker = pipeline('fill-mask', model='bert-base-uncased', device=device)

def augment_text_bert(text, num_augments=1):
    words = text.split()
    if len(words) < 3:
        return [text] * num_augments

    augmented_texts = []

    for _ in range(num_augments):
        num_masks = max(1, int(len(words) * 0.15))
        mask_positions = random.sample(range(len(words)), min(num_masks, len(words)))

        masked_text = words.copy()
        for pos in mask_positions:
            masked_text[pos] = '[MASK]'

        masked_sentence = ' '.join(masked_text)

        try:
            predictions = unmasker(masked_sentence)

            if isinstance(predictions[0], list):
                predictions = predictions[0]

            new_words = words.copy()
            for pos in mask_positions:
                new_words[pos] = predictions[0]['token_str'].strip()

            augmented_texts.append(' '.join(new_words))
        except:
            augmented_texts.append(text)

    return augmented_texts

def augment_dataset_bert(df, text_col='cleaned_text', label_col=None, aug_ratio=0.25):
    num_to_augment = int(len(df) * aug_ratio)

    sample_indices = random.sample(range(len(df)), num_to_augment)

    augmented_rows = []

    for idx in sample_indices:
        original_text = df.iloc[idx][text_col]
        augmented_text = augment_text_bert(original_text, num_augments=1)[0]

        new_row = df.iloc[idx].copy()
        new_row[text_col] = augmented_text
        augmented_rows.append(new_row)

    augmented_df = pd.DataFrame(augmented_rows)
    result_df = pd.concat([df, augmented_df], ignore_index=True)

    return result_df

paths = {
    "dataset1": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset1_downsampled.csv",
    "dataset2": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset2_downsampled.csv",
    "dataset3": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset3_downsampled.csv",
    "dataset4": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset4_downsampled.csv"
}

for name, path in paths.items():
    df = pd.read_csv(path)

    possible_labels = ['label', 'is_depression', 'Suicide']
    label_col = next((col for col in possible_labels if col in df.columns), None)

    dataset_prefix = dataset_mapping[name]

    print(f"Processing {name} for 25% BERT augmentation...")
    df_aug_25 = augment_dataset_bert(df, text_col='cleaned_text', label_col=label_col, aug_ratio=0.25)
    out_path_25 = f"/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25/{dataset_prefix}_bert_25.csv"
    df_aug_25.to_csv(out_path_25, index=False)
    print(f"[SUCCESS] {name} augmented to {len(df_aug_25)} rows (25%) -> saved at: {out_path_25}")

    print(f"Processing {name} for 50% BERT augmentation...")
    df_aug_50 = augment_dataset_bert(df, text_col='cleaned_text', label_col=label_col, aug_ratio=0.50)
    out_path_50 = f"/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/50/{dataset_prefix}_bert_50.csv"
    df_aug_50.to_csv(out_path_50, index=False)
    print(f"[SUCCESS] {name} augmented to {len(df_aug_50)} rows (50%) -> saved at: {out_path_50}")

print("\nAll datasets augmented with BERT (25% and 50%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


Processing dataset1 for 25% BERT augmentation...
[SUCCESS] dataset1 augmented to 1625 rows (25%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25/d1_bert_25.csv
Processing dataset1 for 50% BERT augmentation...
[SUCCESS] dataset1 augmented to 1950 rows (50%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/50/d1_bert_50.csv
Processing dataset2 for 25% BERT augmentation...


Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors


[SUCCESS] dataset2 augmented to 1625 rows (25%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25/d2_bert_25.csv
Processing dataset2 for 50% BERT augmentation...
[SUCCESS] dataset2 augmented to 1950 rows (50%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/50/d2_bert_50.csv
Processing dataset3 for 25% BERT augmentation...
[SUCCESS] dataset3 augmented to 1625 rows (25%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25/d3_bert_25.csv
Processing dataset3 for 50% BERT augmentation...
[SUCCESS] dataset3 augmented to 1950 rows (50%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/50/d3_bert_50.csv
Processing dataset4 for 25% BERT augmentation...
[SUCCESS] dataset4 augmented to 1625 rows (25%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/bert/25/d4_bert_25.csv
Processing dataset4 for 50% BERT augmentation...
[SUCCESS] dataset4 a

back translation

In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import random



In [ ]:
model_name_en_to_fr = 'Helsinki-NLP/opus-mt-en-fr'
model_name_fr_to_en = 'Helsinki-NLP/opus-mt-fr-en'

tokenizer_en_fr = MarianTokenizer.from_pretrained(model_name_en_to_fr)
model_en_fr = MarianMTModel.from_pretrained(model_name_en_to_fr)

tokenizer_fr_en = MarianTokenizer.from_pretrained(model_name_fr_to_en)
model_fr_en = MarianMTModel.from_pretrained(model_name_fr_to_en)

def augment_text_backtranslation(text):
    try:
        inputs = tokenizer_en_fr(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model_en_fr.generate(**inputs)
        french_text = tokenizer_en_fr.decode(translated[0], skip_special_tokens=True)

        inputs_back = tokenizer_fr_en(french_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        back_translated = model_fr_en.generate(**inputs_back)
        english_text = tokenizer_fr_en.decode(back_translated[0], skip_special_tokens=True)

        return english_text.lower()
    except:
        return text

def augment_dataset_backtranslation(df, text_col='cleaned_text', aug_ratio=0.25):
    num_to_augment = int(len(df) * aug_ratio)

    sample_indices = random.sample(range(len(df)), num_to_augment)

    augmented_rows = []

    for idx in sample_indices:
        original_text = df.iloc[idx][text_col]
        augmented_text = augment_text_backtranslation(original_text)

        new_row = df.iloc[idx].copy()
        new_row[text_col] = augmented_text
        augmented_rows.append(new_row)

    augmented_df = pd.DataFrame(augmented_rows)
    result_df = pd.concat([df, augmented_df], ignore_index=True)

    return result_df

paths = {
    "dataset1": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset1_downsampled.csv",
    "dataset2": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset2_downsampled.csv",
    "dataset3": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset3_downsampled.csv",
    "dataset4": "/content/drive/MyDrive/nlpr/smr_nlp_testing/Downsampled/dataset4_downsampled.csv"
}

dataset_mapping = {
    "dataset1": "d1",
    "dataset2": "d2",
    "dataset3": "d3",
    "dataset4": "d4"
}

for name, path in paths.items():
    df = pd.read_csv(path)

    dataset_prefix = dataset_mapping[name]

    print(f"Processing {name} for 25% Back-translation augmentation...")
    df_aug_25 = augment_dataset_backtranslation(df, text_col='cleaned_text', aug_ratio=0.25)
    out_path_25 = f"/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/back_translation/25/{dataset_prefix}_backtranslation_25.csv"
    df_aug_25.to_csv(out_path_25, index=False)
    print(f"[SUCCESS] {name} augmented to {len(df_aug_25)} rows (25%) -> saved at: {out_path_25}")

    print(f"Processing {name} for 50% Back-translation augmentation...")
    df_aug_50 = augment_dataset_backtranslation(df, text_col='cleaned_text', aug_ratio=0.50)
    out_path_50 = f"/content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/back_translation/50/{dataset_prefix}_backtranslation_50.csv"
    df_aug_50.to_csv(out_path_50, index=False)
    print(f"[SUCCESS] {name} augmented to {len(df_aug_50)} rows (50%) -> saved at: {out_path_50}")

print("\nAll datasets augmented with Back-translation (25% and 50%)")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Processing dataset1 for 25% Back-translation augmentation...
[SUCCESS] dataset1 augmented to 1625 rows (25%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/back_translation/25/d1_backtranslation_25.csv
Processing dataset1 for 50% Back-translation augmentation...
[SUCCESS] dataset1 augmented to 1950 rows (50%) -> saved at: /content/drive/MyDrive/nlpr/smr_nlp_testing/data_augumentation/back_translation/50/d1_backtranslation_50.csv
Processing dataset2 for 25% Back-translation augmentation...
